In [ ]:
# Import libraries
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
import re
from transformers import set_seed

/home/norbi/Work/course/ITCS/norbertKrol/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set a fixed seed for reproducibility
SEED = 55
set_seed(SEED)

# Create empty dataframe
df = pd.DataFrame(columns=['id', 'findings', 'impression'])


# Path to the directory with the XML files
document_path = Path('../data/NLMCXR_reports/ecgen-radiology/')

# Loop through the directory
for file in document_path.iterdir():

    # File check 
    if file.suffix == '.xml':

        # Parse file
        medical_data = ET.parse(file)
        medical_data_root = medical_data.getroot()

        # Extract id, findings, and impression
        id_medical = medical_data_root.find('IUXRId').attrib.get('id')
        abstract_data = medical_data_root.find('.//Abstract')
        findings_data = abstract_data.find(".//AbstractText[@Label='FINDINGS']").text
        impression_data = abstract_data.find(".//AbstractText[@Label='IMPRESSION']").text

        # Append data to df
        df = pd.concat([df, pd.DataFrame({'id': [id_medical], 'findings': [findings_data], 'impression': [impression_data]})], ignore_index=True)
        
df       

,id,findings,impression
0,1904,None,Heart size is normal and lungs are clear. Dege...
1,1717,The cardiac contours are normal. Prior granulo...,No acute process.
2,1471,The cardiomediastinal silhouette is within nor...,1. Findings consistent with emphysema with int...
3,2577,Cardiomediastinal silhouette within normal lim...,XXXX XXXX atelectasis/airspace disease.
4,146,None,Heart size and mediastinal contour are stable ...
...,...,...,...
3950,3926,The trachea is midline. The heart XXXX is larg...,1. Low volume study without acute cardiopulmon...
3951,288,Heart size appears within normal limits. Pulmo...,1. No acute cardiopulmonary abnormality.
3952,1477,Heart size within normal limits. No focal alve...,No acute cardiopulmonary findings
3953,2928,Mediastinum is stable. Retrocardiac lucency XX...,1. No acute cardiopulmonary abnormalities. 2. ...


In [3]:
# Convert id to int and sort dataframe
df['id'] = df['id'].astype(int)
df = df.sort_values(by='id', inplace=False).reset_index(drop=True, inplace=False)

# Drop rows with missing values
df.dropna(inplace=True)
df

,id,findings,impression
0,1,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
3,4,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.
5,6,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.
...,...,...,...
3948,3993,The heart is mildly enlarged. Left hemidiaphra...,Borderline cardiomegaly without acute disease.
3949,3994,Similar mild cardiomegaly. Of the pulmonary va...,Mild cardiomegaly with XXXX of early failure.
3950,3995,The cardiomediastinal silhouette and pulmonary...,1. Interval resolution of bibasilar airspace d...
3951,3996,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...


In [4]:
# Regex pattern to change redacted PII with '[REDACTED]'
regex_pattern = r'[Xx]{2,}|[Xx]-X{2,}'

# Regex pattern to remove numbered lists 
regex_pattern_2 = r'\d+\.\s+'

# Regex pattern to remove duplicated periods
regex_pattern_3 = r'\s*\.\s*\.'

# Regex pattern to remove extra spaces
regex_pattern_4 = r'\s+'

In [5]:
# Apply regex patterns
df['findings'] = df['findings'].str.replace(regex_pattern, '[REDACTED]', regex=True).str.replace(regex_pattern_2, '', regex=True).str.replace(regex_pattern_3, '.', regex=True).str.replace(regex_pattern_4, ' ', regex=True)
df['impression'] = df['impression'].str.replace(regex_pattern, '[REDACTED]', regex=True).str.replace(regex_pattern_2, '', regex=True).str.replace(regex_pattern_3, '.', regex=True).str.replace(regex_pattern_4, ' ', regex=True)

df

,id,findings,impression
0,1,The cardiac silhouette and mediastinum size ar...,Normal chest [REDACTED].
1,2,Borderline cardiomegaly. Midline sternotomy [R...,No acute pulmonary findings.
3,4,There are diffuse bilateral interstitial and a...,Bullous emphysema and interstitial fibrosis. P...
4,5,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.
5,6,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.
...,...,...,...
3948,3993,The heart is mildly enlarged. Left hemidiaphra...,Borderline cardiomegaly without acute disease.
3949,3994,Similar mild cardiomegaly. Of the pulmonary va...,Mild cardiomegaly with [REDACTED] of early fai...
3950,3995,The cardiomediastinal silhouette and pulmonary...,Interval resolution of bibasilar airspace dise...
3951,3996,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...


In [6]:
# Check for duplicates
print(f'Number of duplicated findings&impression: {df.duplicated(subset=["findings", "impression"]).sum()}')
print(f'Number of duplicated impression: {df.duplicated(subset=["impression"]).sum()}')
print(f'Number of duplicated findings: {df.duplicated(subset=["findings"]).sum()}')

Number of duplicated findings&impression: 724
Number of duplicated impression: 2060
Number of duplicated findings: 796


In [7]:
# Drop duplicates where findings and impressions are the same, keeping only one row
df = df.drop_duplicates(subset=['findings', 'impression'], keep='first')

# Check for duplicates after cleaning
print(f'After cleaning, number of duplicated findings&impression: {df.duplicated(subset=["findings", "impression"]).sum()}')
print(f'After cleaning, number of duplicated impression: {df.duplicated(subset=["impression"]).sum()}')
print(f'After cleaning, number of duplicated findings: {df.duplicated(subset=["findings"]).sum()}')

After cleaning, number of duplicated findings&impression: 0
After cleaning, number of duplicated impression: 1336
After cleaning, number of duplicated findings: 72


In [8]:
# Export the cleaned dataframe to a CSV file
df.to_csv('../data/cleaned_data.csv', index=False)